Convert `.geojson` vessel annotation files into binary masks in grayscale

In [2]:
from pathlib import Path
import json

import numpy as np
import geopandas as gpd
from shapely.geometry import shape
import cv2
from PIL import Image


def _rasterize_geometries(geometries, height, width):
    mask = np.zeros((height, width), dtype=np.uint8)

    for geom in geometries:
        if geom is None:
            continue

        if geom.geom_type == "Polygon":
            polys = [np.array(geom.exterior.coords)]
        elif geom.geom_type == "MultiPolygon":
            polys = [np.array(p.exterior.coords) for p in geom.geoms]
        else:
            continue

        for poly in polys:
            poly = np.round(poly).astype(np.int32)
            cv2.fillPoly(mask, [poly], 1)

    return mask


def geojson_dir_to_masks(
    geojson_dir: str | Path,
    out_dir: str | Path
):
    """
    Convert a directory of GeoJSON vessel annotations into binary grayscale masks.

    Assumes:
        - GeoJSON coordinates are in pixel space
        - output masks are saved as 0/255 grayscale PNGs
    """

    geojson_dir = Path(geojson_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for geojson_path in geojson_dir.rglob("*.geojson"):

        gdf = gpd.read_file(geojson_path)

        # fallback: infer from bounds (assumes pixel coords)
        minx, miny, maxx, maxy = gdf.total_bounds
        w, h = int(maxx), int(maxy)

        mask = _rasterize_geometries(gdf.geometry, h, w)

        out_path = out_dir / geojson_path.name.replace(".geojson", "_mask.png")
        Image.fromarray(mask * 255).save(out_path)

In [3]:
geojson_dir = "/Users/sophiali/Desktop/chip-stroma-analysis/data/raw/raw_masks"
out_dir = "/Users/sophiali/Desktop/chip-stroma-analysis/data/raw/binary_masks"

geojson_dir_to_masks(geojson_dir, out_dir)